# Exercise 2 — Industrial transfer learning with MVTec Capsule

This notebook is the more independent part of the exercise.

You now apply the full transfer-learning pipeline to an industrially motivated dataset: **MVTec Capsule**.

Important teaching note:

The original MVTec AD benchmark is an anomaly-detection benchmark, where training is usually done on defect-free images. In this exercise, we use a **supervised teaching split** with normal and abnormal examples. This is not the official benchmark protocol. The goal is to learn how transfer-learning choices affect an industrial binary-classification pipeline.

## What you will do

Compared with the CIFAR-10 notebook, you must make more decisions yourself:

1. configure the industrial dataset,
2. choose preprocessing / augmentation settings,
3. choose a CNN backbone and transfer-learning strategy,
4. train for multiple epochs,
5. compare hyperparameters,
6. inspect confusion matrix and error cases,
7. write an industrial interpretation.

You are expected to look up PyTorch/TorchVision documentation when needed.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from cvis_ml.config import DatasetConfig, ModelConfig, TrainConfig
from cvis_ml.data import (
    MVTecCapsuleDataModule,
    make_class_weights_from_counts,
    auto_pin_memory,
)
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import run_transfer_experiment, resolve_device
from cvis_ml.visualization import (
    show_batch, plot_history, show_confusion_matrix,
    show_misclassified, results_table
)

torch.manual_seed(42)
np.random.seed(42)

device = resolve_device("auto")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("DataLoader pin_memory will be:", auto_pin_memory())

## Part A — Configure and inspect the industrial dataset

Start with a moderate image size and a small batch size.

Recommended settings:

- `image_size = 128`
- `batch_size = 16`
- `max_train_samples = 240`
- `max_val_samples = 120`
- `num_workers = 0` on Windows / CPU
- `augment = True`

The Hugging Face dataset may take some time to download on the first run.

In [ ]:
mvtec_cfg = DatasetConfig(
    name="mvtec_capsule_teaching_split",
    data_root=str(repo_root / "data_cache"),
    image_size=...,          # TODO
    batch_size=...,          # TODO
    max_train_samples=...,   # TODO
    max_val_samples=...,     # TODO
    num_workers=...,         # TODO
    seed=42,
    augment=...,             # TODO
)

print(mvtec_cfg)

In [ ]:
# Create data module and load data.

mvtec_dm = MVTecCapsuleDataModule(
    root=mvtec_cfg.data_root,
    image_size=mvtec_cfg.image_size,
    batch_size=mvtec_cfg.batch_size,
    num_workers=mvtec_cfg.num_workers,
    max_train_samples=mvtec_cfg.max_train_samples,
    max_val_samples=mvtec_cfg.max_val_samples,
    seed=mvtec_cfg.seed,
    augment=mvtec_cfg.augment,
)

mvtec_data = mvtec_dm.setup()

print("Classes:", mvtec_data.class_names)
print("Number of classes:", mvtec_data.num_classes)
print("Train class counts:", mvtec_data.train_counts)
print("Validation class counts:", mvtec_data.val_counts)
print("Train batches:", len(mvtec_data.train_loader))
print("Validation batches:", len(mvtec_data.val_loader))

show_batch(mvtec_data.train_loader, mvtec_data.class_names, n=8)

### Task A1 — Industrial dataset interpretation 🟢

Answer briefly:

1. Is the dataset balanced?
2. Why might a normal/abnormal classification split be easier than the official anomaly-detection setup?
3. Why should we still be careful when interpreting validation results?

In [ ]:
answer_A1 = """


"""
print(answer_A1)

## Part B — Choose an industrial transfer-learning baseline

For the industrial task, start with a model that is not too heavy.

Recommended first baseline:

- architecture: `resnet18`
- strategy: `frozen`
- epochs: `5`
- learning rate: `1e-3`

You may later compare partial fine-tuning or MobileNetV3-Small.

In [ ]:
# Choose your first industrial baseline.

architecture = ...  # TODO: e.g. "resnet18" or "mobilenet_v3_small"
strategy = ...      # TODO: "frozen", "partial", or "full"

model_cfg = ModelConfig(
    architecture=architecture,
    strategy=strategy,
    pretrained=True,
    num_classes=mvtec_data.num_classes,
)

industrial_model = TransferModelFactory.create(
    architecture=model_cfg.architecture,
    strategy=model_cfg.strategy,
    pretrained=model_cfg.pretrained,
    num_classes=model_cfg.num_classes,
)

total_params, trainable_params = count_parameters(industrial_model)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
pd.DataFrame(describe_trainable_parameters(industrial_model, max_rows=20))

## Part C — Train the industrial baseline

Train for more than one epoch. Start with **5 epochs**.

If you use partial or full fine-tuning, reduce the learning rate. For example:

| strategy | suggested learning rate |
|---|---:|
| frozen | `1e-3` |
| partial | `1e-4` |
| full | `3e-5` to `1e-4` |

Optional: enable class weights if the class counts are imbalanced.

In [ ]:
# Optional: class weights for imbalanced binary classification.
# Use this later if you want to compare weighted vs unweighted training.

use_class_weights = False

class_weights = None
if use_class_weights:
    class_weights = make_class_weights_from_counts(
        mvtec_data.train_counts,
        num_classes=mvtec_data.num_classes,
    )
    print("Class weights:", class_weights)

train_cfg = TrainConfig(
    epochs=...,             # TODO: start with 5
    learning_rate=...,      # TODO: strategy-dependent
    weight_decay=1e-4,
    device="auto",
    use_class_weights=use_class_weights,
    max_batches_per_epoch=None,
    scheduler=None,
)

industrial_result = run_transfer_experiment(
    name=f"mvtec_capsule_{architecture}_{strategy}",
    model=industrial_model,
    train_loader=mvtec_data.train_loader,
    val_loader=mvtec_data.val_loader,
    epochs=train_cfg.epochs,
    learning_rate=train_cfg.learning_rate,
    weight_decay=train_cfg.weight_decay,
    device=train_cfg.device,
    class_weights=class_weights,
    max_batches_per_epoch=train_cfg.max_batches_per_epoch,
    scheduler=train_cfg.scheduler,
)

plot_history(industrial_result.history, title=industrial_result.name)
show_confusion_matrix(
    industrial_result.y_true,
    industrial_result.y_pred,
    mvtec_data.class_names,
    title=industrial_result.name,
)

### Short interpretation C 🟢 / 🟡

Answer:

1. How did validation accuracy and macro-F1 evolve?
2. Did the model confuse normal and abnormal images?
3. Which error is more dangerous in industrial inspection: false normal or false abnormal?
4. Does the result look trustworthy enough for a real inspection system?

In [ ]:
answer_C = """


"""
print(answer_C)

## Part D — Hyperparameter and preprocessing experiment 🟡

Now define at least two additional experiments.

You may vary:

- learning rate,
- number of epochs,
- architecture,
- transfer strategy,
- augmentation on/off,
- class weights.

Good experiment ideas:

1. frozen ResNet18, 5 epochs, lr = 1e-3
2. frozen ResNet18, 8 epochs, lr = 1e-3
3. partial ResNet18, 5 epochs, lr = 1e-4
4. MobileNetV3-Small, frozen, 5 epochs, lr = 1e-3
5. same model with `augment=False`
6. same model with class weights

Do not run everything during the live exercise. Choose two or three meaningful comparisons.

In [ ]:
def run_mvtec_setting(
    name,
    architecture,
    strategy,
    epochs,
    learning_rate,
    augment=True,
    use_class_weights=False,
    image_size=128,
    max_train_samples=240,
    max_val_samples=120,
):
    """Create data, model, and trainer for one MVTec Capsule experiment."""
    dm = MVTecCapsuleDataModule(
        root=mvtec_cfg.data_root,
        image_size=image_size,
        batch_size=mvtec_cfg.batch_size,
        num_workers=mvtec_cfg.num_workers,
        max_train_samples=max_train_samples,
        max_val_samples=max_val_samples,
        seed=mvtec_cfg.seed,
        augment=augment,
    )
    data = dm.setup()
    model_x = TransferModelFactory.create(
        architecture=architecture,
        strategy=strategy,
        pretrained=True,
        num_classes=data.num_classes,
    )
    weights = None
    if use_class_weights:
        weights = make_class_weights_from_counts(data.train_counts, num_classes=data.num_classes)

    result = run_transfer_experiment(
        name=name,
        model=model_x,
        train_loader=data.train_loader,
        val_loader=data.val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=1e-4,
        device="auto",
        class_weights=weights,
        max_batches_per_epoch=None,
    )
    return result

In [ ]:
# TODO:
# Define at least two meaningful experiment settings.
# You should be able to justify what each setting tests.

mvtec_experiment_settings = [
    {"name": "mvtec_resnet18_frozen_ep8", "architecture": "resnet18", "strategy": "frozen", "epochs": 8, "learning_rate": 1e-3, "augment": True, "use_class_weights": False},
    # Add at least one more:
    # {"name": ..., "architecture": ..., "strategy": ..., "epochs": ..., "learning_rate": ..., "augment": ..., "use_class_weights": ...},
]

industrial_results = [industrial_result]

for cfg in mvtec_experiment_settings:
    print("\nRunning", cfg)
    res = run_mvtec_setting(**cfg)
    industrial_results.append(res)

results_table(industrial_results)

## Part E — Error analysis 🟡

Choose one trained model and inspect misclassified validation images.

This is especially important in industrial tasks because the two error types do not have the same meaning:

- false abnormal: unnecessary alarm / rework
- false normal: missed defect / quality escape

In [ ]:
# The variable `industrial_model` is the first model trained in this notebook.
# If you want to inspect a model from a later experiment, rerun it or adapt the helper to return the model too.

show_misclassified(
    industrial_model,
    mvtec_data.val_loader,
    mvtec_data.class_names,
    device=resolve_device("auto"),
    n=8,
    max_batches=20,
)

## Part F — Documentation lookup and design recommendation 🔴

Look up one PyTorch or TorchVision documentation page relevant to one of your experiments.

Possible topics:

- `torch.optim.AdamW`
- `torchvision.transforms.ColorJitter`
- `torchvision.transforms.RandomRotation`
- `torchvision.models.resnet18`
- `torchvision.models.mobilenet_v3_small`

Then write a final industrial recommendation.

In [ ]:
final_industrial_report = """
1. Best-performing setting:


2. Most important hyperparameter effect:


3. Preprocessing / augmentation effect:


4. Industrial error analysis:
False abnormal means ...
False normal means ...

5. What I would do before deployment:


6. Documentation page I looked up and what I learned:


"""
print(final_industrial_report)